# 🔄 Notebook 4: Read Replicas

When a single database can't handle the read load, distribute reads across multiple servers using read replicas.

## Learning Objectives

By the end of this notebook, you'll understand:
- Leader-follower replication
- Synchronous vs asynchronous replication
- Handling replication lag
- Read-after-write consistency

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [ ]:
import time
import random
from dataclasses import dataclass
from typing import Optional
from concurrent.futures import ThreadPoolExecutor

print("✅ Ready to learn about read replicas!")

## 🏗️ Leader-Follower Architecture

In [ ]:
print("🏗️ Leader-Follower Replication")
print("=" * 60)
print("""
                    ┌─────────────────┐
                    │  Application    │
                    └────────┬────────┘
                             │
              ┌──────────────┴──────────────┐
              │                             │
        WRITES│                       READS │
              ▼                             ▼
    ┌─────────────────┐         ┌─────────────────┐
    │     LEADER      │         │   FOLLOWERS     │
    │    (Primary)    │────────►│   (Replicas)    │
    │                 │ replicate│                 │
    │  • All writes   │         │  • Read only    │
    │  • Source of    │         │  • Multiple     │
    │    truth        │         │    servers      │
    └─────────────────┘         └─────────────────┘

KEY POINTS:
─────────────────────────────────────────────────────────────
• Writes ONLY go to leader
• Leader replicates changes to followers
• Reads distributed across ALL followers
• Add more followers = handle more reads
""")

## 🔄 Simulating Replication

In [ ]:
@dataclass
class Database:
    name: str
    is_leader: bool
    data: dict
    replication_lag_ms: float = 0
    
    def read(self, key: str) -> Optional[str]:
        time.sleep(0.001)
        return self.data.get(key)
    
    def write(self, key: str, value: str) -> bool:
        if not self.is_leader:
            raise Exception("Cannot write to replica!")
        time.sleep(0.002)
        self.data[key] = value
        return True

class ReplicatedDatabase:
    def __init__(self, num_replicas: int = 3, replication_lag_ms: float = 50):
        self.leader = Database("leader", True, {})
        self.replicas = [
            Database(f"replica-{i}", False, {}, replication_lag_ms)
            for i in range(num_replicas)
        ]
        self.replication_lag_ms = replication_lag_ms
        self.pending_replications = []
    
    def write(self, key: str, value: str):
        self.leader.write(key, value)
        write_time = time.time()
        self.pending_replications.append((key, value, write_time))
        return True
    
    def _apply_replications(self):
        current_time = time.time()
        still_pending = []
        
        for key, value, write_time in self.pending_replications:
            elapsed_ms = (current_time - write_time) * 1000
            if elapsed_ms >= self.replication_lag_ms:
                for replica in self.replicas:
                    replica.data[key] = value
            else:
                still_pending.append((key, value, write_time))
        
        self.pending_replications = still_pending
    
    def read_from_replica(self, key: str) -> tuple:
        self._apply_replications()
        replica = random.choice(self.replicas)
        value = replica.read(key)
        return value, replica.name
    
    def read_from_leader(self, key: str) -> tuple:
        value = self.leader.read(key)
        return value, "leader"

db = ReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
print("✅ Replicated database created with 3 replicas")
print(f"   Simulated replication lag: 100ms")

In [ ]:
print("🔄 Demonstrating Replication Lag")
print("=" * 60)

db.write("user:1:name", "Alice")
print("\n✏️ Wrote 'Alice' to leader")

print("\n📖 Reading immediately after write:")
immediate = []
for i in range(3):
    value, source = db.read_from_replica("user:1:name")
    immediate.append(value)
    status = "✅" if value else "❌ STALE"
    print(f"   Read from {source}: {value or 'None'} {status}")

print("\n⏳ Waiting for replication (150ms)...")
time.sleep(0.15)

print("\n📖 Reading after replication lag:")
settled = []
for i in range(3):
    value, source = db.read_from_replica("user:1:name")
    settled.append(value)
    status = "✅" if value else "❌ STALE"
    print(f"   Read from {source}: {value or 'None'} {status}")

# The whole point of this notebook is that the FIRST reads are wrong. If the
# simulation ever gets "too fast" and the replicas answer correctly straight
# away, this cell must fail loudly rather than quietly teach nothing.
assert all(v is None for v in immediate), (
    f"expected every read inside the {db.replication_lag_ms:.0f}ms lag window to "
    f"miss, got {immediate} — the lag demo is no longer demonstrating lag"
)
assert all(v == "Alice" for v in settled), (
    f"expected the replicas to have caught up after 150ms, got {settled}"
)
print("\n✅ Reproduced: reads inside the lag window saw nothing, reads after it saw 'Alice'.")

## ⚠️ The Read-After-Write Problem

In [ ]:
print("⚠️ Read-After-Write Consistency Problem")
print("=" * 60)
print("""
SCENARIO: User updates their profile
─────────────────────────────────────────────────────────────

1. User changes name from "Alice" to "Alicia"
   └─► Write goes to LEADER

2. Page refreshes to show updated profile
   └─► Read goes to REPLICA (still has "Alice"!)

3. User sees OLD name "Alice" 😱
   └─► Thinks the update failed!

─────────────────────────────────────────────────────────────
""")

db2 = ReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
db2.write("user:1:name", "Alice")
time.sleep(0.15)

print("\nSimulating the problem:")
print("1. User updates name to 'Alicia'...")
db2.write("user:1:name", "Alicia")

print("2. Page refreshes, reading from replica...")
value, source = db2.read_from_replica("user:1:name")
leader_value, _ = db2.read_from_leader("user:1:name")
print(f"   Got: '{value}' from {source}")
print(f"   Meanwhile the leader already says: '{leader_value}'")
print(f"   ❌ User sees old name!")

# This is the real thing, not a description of it: the replica genuinely still
# holds the PREVIOUS value, so the user's own update looks like it never landed.
assert leader_value == "Alicia", f"leader must already be correct, got {leader_value!r}"
assert value == "Alice", (
    f"expected the replica to still serve the stale 'Alice', got {value!r} — "
    f"if this stops failing, the read-after-write lesson has evaporated"
)

## 🛡️ Solutions for Read-After-Write

In [ ]:
print("🛡️ Solutions for Read-After-Write Consistency")
print("=" * 60)
print("""
SOLUTION 1: Read Your Own Writes from Leader
─────────────────────────────────────────────────────────────
• After write, read from leader for that user's session
• Track "last write timestamp" per user
• If recent write, read from leader; else read from replica

SOLUTION 2: Synchronous Replication
─────────────────────────────────────────────────────────────
• Wait for at least one replica to confirm
• Slower writes, but guaranteed consistency
• Trade-off: higher latency for writes

SOLUTION 3: Client-Side Optimistic Updates
─────────────────────────────────────────────────────────────
• Update UI immediately after write
• Don't re-fetch from database
• Simple but doesn't solve server-side reads

SOLUTION 4: Causal Consistency Token
─────────────────────────────────────────────────────────────
• Return write timestamp with response
• Client sends timestamp with next read
• Server waits for replica to catch up to that timestamp
""")

In [ ]:
class SmartReplicatedDatabase(ReplicatedDatabase):
    """Pins a user to the LEADER for a while after that user writes.

    `sticky_window_ms` is the knob that decides whether this works. It has to
    cover the WORST replication lag you actually observe, not the average one —
    see the next cell for what happens when it doesn't.
    """

    def __init__(self, *args, sticky_window_ms: Optional[float] = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.user_last_writes = {}
        # Default: 2x the expected lag, i.e. a safety factor of two.
        self.sticky_window_ms = (
            sticky_window_ms if sticky_window_ms is not None
            else self.replication_lag_ms * 2
        )

    def write_for_user(self, user_id: str, key: str, value: str):
        self.write(key, value)
        self.user_last_writes[user_id] = time.time()
        return True

    def read_for_user(self, user_id: str, key: str) -> tuple:
        last_write = self.user_last_writes.get(user_id, 0)
        time_since_write = (time.time() - last_write) * 1000

        if time_since_write < self.sticky_window_ms:
            return self.read_from_leader(key)
        else:
            return self.read_from_replica(key)

print("🛡️ Smart Database with Read-Your-Own-Writes")
print("=" * 60)

smart_db = SmartReplicatedDatabase(num_replicas=3, replication_lag_ms=100)
smart_db.write("user:1:name", "Alice")
time.sleep(0.15)

print(f"\nSticky window: {smart_db.sticky_window_ms:.0f}ms "
      f"(2x the {smart_db.replication_lag_ms:.0f}ms replication lag)")

print("\n1. User updates name to 'Alicia'...")
smart_db.write_for_user("user-1", "user:1:name", "Alicia")

print("2. Page refreshes with smart routing...")
value, source = smart_db.read_for_user("user-1", "user:1:name")
print(f"   Got: '{value}' from {source}")
print(f"   ✅ Routed to leader for recent writer!")

assert source == "leader", f"the writer must be routed to the leader, got {source}"
assert value == "Alicia", f"the writer must see their own write, got {value!r}"

print("\n3. Another user reads same data...")
other_value, other_source = smart_db.read_for_user("user-2", "user:1:name")
print(f"   Got: '{other_value}' from {other_source}")
print(f"   ✅ Routed to replica (didn't write recently)")

assert other_source.startswith("replica"), (
    f"a non-writer should be served from a replica, got {other_source}"
)
assert other_value == "Alice", (
    f"expected user-2 to still see the pre-update value, got {other_value!r}"
)

print()
print("⚖️  Be honest about what we just bought: user-2 IS reading stale data.")
print("    Read-your-own-writes fixes the ONE case users notice (their own edit")
print("    appearing to vanish). Everybody else still gets eventual consistency —")
print("    that's the whole trade, and it's why replicas stay cheap.")

### ⚠️ The routing rule is only as good as its window

`sticky_window_ms` is a *guess* about how far behind the replicas are. Guess low
and read-your-own-writes silently stops working — for exactly the users who just
wrote. It's a nasty bug because it only appears under load, when lag grows past
the window you tuned on a quiet afternoon.


In [ ]:
print("⚠️ What happens when the sticky window is TOO SHORT")
print("=" * 60)

# Replicas are 100ms behind, but we only pin the writer to the leader for 20ms.
bad_db = SmartReplicatedDatabase(num_replicas=3, replication_lag_ms=100,
                                 sticky_window_ms=20)
bad_db.write("user:1:name", "Alice")
time.sleep(0.15)

bad_db.write_for_user("user-1", "user:1:name", "Alicia")
time.sleep(0.05)   # 50ms later: past the 20ms window, still inside the 100ms lag
value, source = bad_db.read_for_user("user-1", "user:1:name")

print(f"\n   sticky window : {bad_db.sticky_window_ms:.0f}ms")
print(f"   actual lag    : {bad_db.replication_lag_ms:.0f}ms")
print(f"   read 50ms after the write → {source} returned '{value}'")

assert not source.startswith("leader"), "the 20ms window should already have expired"
assert value == "Alice", (
    f"expected the writer's own update to be missing, got {value!r}"
)
print("   ❌ The user's own write vanished again — the guarantee is BROKEN.")

print()
print("👉 Rule of thumb: sticky_window >= the MAX replication lag you observe,")
print("   not the mean. In Postgres that number is pg_stat_replication.replay_lag")
print("   (see the section at the end) — alert on it, because a window tuned to")
print("   yesterday's p50 stops holding during today's spike.")
print()
print("👉 Solution 4 removes the guess entirely: hand the client the write's LSN,")
print("   and make the replica wait until it has replayed that LSN before")
print("   answering. Slower reads, but no magic number to get wrong.")

## 📊 Sync vs Async Replication

In [ ]:
print("📊 Synchronous vs Asynchronous Replication")
print("=" * 60)
print("""
ASYNCHRONOUS (Default)
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Response
                  │
                  └──► Replicas (eventually)

• Write returns immediately
• Replicas catch up in background
• Risk: Data loss if leader fails before replication
• Pro: Faster writes

─────────────────────────────────────────────────────────────

SYNCHRONOUS
─────────────────────────────────────────────────────────────
    Client ──► Leader ──► Replicas ──► ACK ──► Response

• Write waits for replica confirmation
• Guaranteed to be on at least 2 servers
• Risk: Slower writes, replica failure blocks writes
• Pro: No data loss

─────────────────────────────────────────────────────────────

SEMI-SYNCHRONOUS (Common in production)
─────────────────────────────────────────────────────────────
• Wait for ONE replica to confirm
• Other replicas are async
• Balance between safety and speed
""")

## 📈 Scaling with Replicas — measured, not asserted

It is very easy to write `capacity = num_replicas * 10_000` in a slide and call
it scaling. That number is a *claim*, not a result. To actually earn it we have
to model the thing that makes a database fall over: **queueing**.

Each replica here is one server that can run **one read at a time**, taking
`SERVICE_MS` milliseconds. Requests arrive as a Poisson stream, the load
balancer picks a replica, and if that replica is busy the request *waits*. Then
we search for the highest offered load whose **p99 still fits our SLO** — that
is the replica set's real capacity.


In [ ]:
import random

SERVICE_MS = 2.0     # how long ONE replica takes to serve ONE read
SLO_P99_MS = 20.0    # the latency we promise users
SIM_SECONDS = 3.0    # simulated wall-clock per run


def arrival_times(offered_rps: float, seconds: float = SIM_SECONDS) -> list:
    """Poisson arrivals, in milliseconds. Seeded, so every run is identical."""
    rng = random.Random(1234)
    t, end, out = 0.0, seconds * 1000.0, []
    while True:
        t += rng.expovariate(offered_rps / 1000.0)
        if t >= end:
            return out
        out.append(t)


def run(num_replicas: int, offered_rps: float, policy: str = "random") -> dict:
    """Replay the same arrival stream against N replicas and record latency.

    `free_at[i]` is when replica i finishes whatever it is currently running.
    A request that lands on a busy replica QUEUES — that queue is the whole
    reason a database has a capacity limit at all.
    """
    rng = random.Random(99)
    free_at = [0.0] * num_replicas
    latencies = []

    for t in arrival_times(offered_rps):
        if policy == "random":
            i = rng.randrange(num_replicas)              # dumb round-robin-ish LB
        else:
            i = min(range(num_replicas), key=lambda k: free_at[k])  # least-busy
        finish = max(t, free_at[i]) + SERVICE_MS
        free_at[i] = finish
        latencies.append(finish - t)                     # queue wait + service

    latencies.sort()
    n = len(latencies)
    return {"n": n, "p50": latencies[n // 2],
            "p99": latencies[min(n - 1, int(n * 0.99))]}


def capacity(num_replicas: int, policy: str = "random") -> float:
    """Highest offered rps whose p99 still meets the SLO. Binary search."""
    lo, hi = 0.0, 60_000.0
    for _ in range(15):
        mid = (lo + hi) / 2
        if run(num_replicas, mid, policy)["p99"] <= SLO_P99_MS:
            lo = mid
        else:
            hi = mid
    return lo


print("📈 Scaling Reads with Replicas")
print("=" * 68)
print(f"service time {SERVICE_MS:.0f}ms/read · SLO p99 <= {SLO_P99_MS:.0f}ms\n")

baseline = capacity(1)
print(f"{'replicas':>8} {'capacity (rps)':>16} {'vs 1 replica':>14} {'least-busy LB':>16}")
print("-" * 68)

measured = {}
for n in [1, 2, 4, 8]:
    cap_random = capacity(n, "random")
    cap_leastbusy = capacity(n, "least-busy")
    measured[n] = (cap_random, cap_leastbusy)
    print(f"{n:>8} {cap_random:>16,.0f} {cap_random / baseline:>13.2f}x "
          f"{cap_leastbusy:>15,.0f}")

mult_8 = measured[8][0] / baseline
print(f"\n📊 Measured multiplier at 8 replicas: {mult_8:.2f}x "
      f"(a perfect world would give 8.00x)")

# If this ever stops being roughly linear, the lab's central claim is wrong and
# should fail here rather than in a reader's head.
assert mult_8 >= 6.0, (
    f"read replicas are supposed to scale reads roughly linearly; measured only "
    f"{mult_8:.2f}x at 8 replicas"
)
assert measured[8][1] > measured[8][0], (
    "least-busy routing should beat random routing; if not, the queue model is broken"
)

lb_gain = measured[8][1] / measured[8][0] - 1
print(f"   Random routing leaves {lb_gain * 100:.0f}% of that capacity on the floor:")
print( "   N independent queues idle while one of them has a backlog. A least-busy")
print( "   (or least-connections) balancer shares one logical queue and wins it back.")

print("\n" + "=" * 68)
print("Same replica sets under a FIXED 1,200 rps of offered load:")
print("-" * 68)
overload = {n: run(n, 1200.0) for n in [1, 2, 4, 8]}
for n, res in overload.items():
    verdict = "✅ within SLO" if res["p99"] <= SLO_P99_MS else "💀 drowning"
    print(f"   {n:>2} replicas → p50 {res['p50']:8.2f}ms   p99 {res['p99']:9.2f}ms   {verdict}")

assert overload[1]["p99"] > 50 * overload[8]["p99"], (
    f"one replica should be visibly drowning at 1200 rps; got p99 "
    f"{overload[1]['p99']:.1f}ms vs {overload[8]['p99']:.1f}ms with 8"
)
assert overload[1]["p99"] > SLO_P99_MS and overload[8]["p99"] <= SLO_P99_MS

print("\n💡 Note what did NOT change: p50 at low load is ~SERVICE_MS no matter how")
print("   many replicas you have. Replicas buy you THROUGHPUT, not a faster query.")
print("   The only reason latency collapses above is that the queue drained.")
print()
print("⚠️  What this toy leaves out, and production doesn't:")
print("   • replicas also spend CPU replaying WAL — reads compete with replication")
print("   • a replica that falls behind still answers, just with older data")
print("   • writes do NOT scale here at all: every one still goes to the leader")
print("   • losing the leader costs a failover, during which writes stop")

## 🧪 Quick Quiz

1. **Why can't you write to a replica?**

2. **What's the read-after-write problem?**

3. **When would you use synchronous replication?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why no writes to replicas:")
print("   - Creates conflict (which write wins?)")
print("   - Leader is source of truth")
print("   - Replication is one-way: leader → follower")
print()
print("2. Read-after-write problem:")
print("   - User writes to leader")
print("   - Immediately reads from replica")
print("   - Replica hasn't received write yet")
print("   - User sees stale (old) data")
print()
print("3. When to use sync replication:")
print("   - Critical data (financial, medical)")
print("   - When data loss is unacceptable")
print("   - Willing to accept slower writes")

## 🐘 Doing This for Real in PostgreSQL

So far we simulated replication in Python. That was great for understanding
the *concepts*, but in production you use your database's built-in replication.
PostgreSQL offers two flavours:

### 1. Streaming replication (physical)

The replica is a **byte-for-byte copy** of the primary. Postgres ships its
Write-Ahead Log (WAL) to the replica, which replays it.

**Minimal setup** (primary's `postgresql.conf`):

```conf
wal_level = replica            # needed so the WAL contains enough info
max_wal_senders = 10           # how many replicas can stream at once
```

**On the primary** — create a replication user:

```sql
CREATE ROLE repuser WITH REPLICATION LOGIN PASSWORD 'secret';
```

**On the replica** — clone the primary, then start it as a standby:

```bash
# One-time: copy the entire data directory from primary.
pg_basebackup -h primary-host -U repuser -D /var/lib/postgresql/data -Fp -Xs -P -R
# The -R flag writes a standby.signal file so Postgres boots in replica mode.

# Then just start Postgres. It will connect to the primary and stream WAL.
```

**Pros**: automatic, whole-cluster, lowest lag.
**Cons**: replica is read-only, must be same major version, can't replicate a subset.

### 2. Logical replication (selective)

Replicate *specific tables* between databases — even across major versions.
Uses `PUBLICATION` (on primary) + `SUBSCRIPTION` (on replica).

```sql
-- On primary:
CREATE PUBLICATION posts_pub FOR TABLE posts, users;

-- On replica (different cluster, possibly different version):
CREATE SUBSCRIPTION posts_sub
  CONNECTION 'host=primary-host dbname=scaling_demo user=repuser password=secret'
  PUBLICATION posts_pub;
```

**Great for**: analytics replicas, zero-downtime upgrades, replicating only the
hot tables to a cheaper read-only instance.

### 📏 Monitoring replication lag (the #1 thing you'll debug)

Once replication is running, the single most important query is:

```sql
-- Run this ON THE PRIMARY:
SELECT
    client_addr,
    state,
    sent_lsn,                 -- how much WAL we've sent
    replay_lsn,               -- how much the replica has applied
    pg_wal_lsn_diff(sent_lsn, replay_lsn) AS lag_bytes,
    write_lag, flush_lag, replay_lag  -- time-based lag
FROM pg_stat_replication;
```

If `replay_lag` grows to seconds or minutes, your replica is falling behind —
that's when read-after-write problems start biting users.

### 🧰 Who actually sets this up?

In real life, almost nobody configures streaming replication by hand. You use:

| Tool | What it gives you |
|------|-------------------|
| **Amazon RDS / Aurora** | One click: "Create read replica" |
| **Google Cloud SQL** | Same, via `--master-instance-name` flag |
| **Patroni** + etcd | Self-managed HA with automatic failover |
| **Bitnami Postgres-HA Helm chart** | Kubernetes-native primary + replicas |

The *concepts* in this notebook (leader/follower, lag, read-your-own-writes)
are what you apply to each of these — they all have the same failure modes.


## 📚 Summary

### Key Takeaways

1. **Leader handles writes, replicas handle reads** - simple split
2. **Replication lag is unavoidable** - plan for it
3. **Read-your-own-writes** - route recent writers to the leader, using a\n   sticky window >= your MAX observed lag (a too-short window silently\n   reintroduces the bug)
4. **Add replicas to scale READS** - roughly linear (we measured ~7.5x at\n   8 replicas with random routing, ~9.7x with least-busy). Writes do not\n   scale: they all still land on the one leader.
5. **Sync vs async** - trade-off between safety and speed

### Next Up

In **Notebook 5**, we'll learn about application caching:
- Redis as a cache layer
- TTL strategies
- Cache invalidation